# 🔬 NB3 · Case Study: "100% accuracy" · Caso de estudio: "100% de exactitud"
**Research Methods & Scientific Integrity in AI · Session 5 · UNMSM Deep Tech Doctorate · Dr. Loveleen Gaur**  
*Métodos de Investigación e Integridad Científica en IA · Sesión 5*


**EN:** Instructor walk-through (about 15 min). We take a published claim, check it, fix it and write an honest report. Students re-run it at home.  
**ES:** Recorrido guiado del docente (unos 15 min). Tomamos una afirmación publicada, la revisamos, la corregimos y escribimos un informe honesto. Los estudiantes lo ejecutan en casa.

> ⚠️ **EN:** The "paper" is invented for teaching. The data, the code and every number are real and computed live.  
> **ES:** El "artículo" es inventado para enseñar. Los datos, el código y cada número son reales y se calculan en vivo.  
> Data · Datos: Wisconsin Diagnostic Breast Cancer, 569 patients, 30 features ([UCI](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)).

| Step · Paso | EN | ES |
|---|---|---|
| 1 | Read the claim | Leer la afirmación |
| 2 | Re-run it | Volver a ejecutarla |
| 3 | Check for problems | Buscar problemas |
| 4 | Fix and re-run | Corregir y volver a ejecutar |
| 5 | Write an honest report | Escribir un informe honesto |

> **🇪🇸 Traducción al español**
>
> # 🔬 NB3 · Caso de estudio: "100 % de exactitud"
> **Métodos de Investigación e Integridad Científica en IA · Sesión 5 · Doctorado en Deep Tech UNMSM · Dra. Loveleen Gaur**
>
> Recorrido guiado del docente (unos 15 min). Tomamos una afirmación publicada, la revisamos, la corregimos y escribimos un informe honesto. Los estudiantes lo vuelven a ejecutar en casa.
>
> ⚠️ El "artículo" es inventado con fines didácticos. Los datos, el código y cada número son reales y se calculan en vivo.
> Datos: Wisconsin Diagnostic Breast Cancer, 569 pacientes, 30 variables ([UCI](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)).
>
> | Paso | Descripción |
> |---|---|
> | 1 | Leer la afirmación |
> | 2 | Volver a ejecutarla |
> | 3 | Buscar problemas |
> | 4 | Corregir y volver a ejecutar |
> | 5 | Escribir un informe honesto |

In [ ]:
# Importa platform (info del sistema), warnings, NumPy, pandas y scikit-learn
import platform, warnings, numpy as np, pandas as pd, sklearn
# Silencia las advertencias
warnings.filterwarnings("ignore")
# Carga el conjunto de datos de cáncer de mama de Wisconsin
from sklearn.datasets import load_breast_cancer
# División entrenamiento/prueba, validación cruzada estratificada repetida y su evaluación
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_val_score
# Escalador que estandariza las variables
from sklearn.preprocessing import StandardScaler
# Selección de las k mejores variables con la prueba F
from sklearn.feature_selection import SelectKBest, f_classif
# Red neuronal multicapa
from sklearn.neural_network import MLPClassifier
# Regresión logística (modelo simple de comparación)
from sklearn.linear_model import LogisticRegression
# Clasificador "tonto" que siempre predice la clase mayoritaria
from sklearn.dummy import DummyClassifier
# Encadena pasos en un Pipeline
from sklearn.pipeline import make_pipeline
# Métrica de exactitud
from sklearn.metrics import accuracy_score
# Imprime las versiones de Python y scikit-learn
print("Python", platform.python_version(), "| scikit-learn", sklearn.__version__)
# Carga variables (X) y etiquetas (y); 0 = maligno, 1 = benigno
X, y = load_breast_cancer(return_X_y=True)
# Imprime el número de pacientes, de variables y el conteo de malignos y benignos
print("Patients / Pacientes:", X.shape[0], "| Features / Variables:", X.shape[1],
      # Continuación del print: conteo de pacientes malignos (y == 0) y benignos (y == 1)
      "| Malignant / Malignos:", (y == 0).sum(), "| Benign / Benignos:", (y == 1).sum())

## Step 1 · The claim · La afirmación
> *"Our neural network (DeepOnco-MLP) detects malignant tumours with **100% accuracy**. We balanced the classes by copying minority cases (oversampling), scaled the data, kept the 10 best features, and tested on 20% of the data (seed 12)."*

**EN:** It sounds careful. **Do you believe it?** (Quick vote.)  
**ES:** Suena cuidadoso. **¿Lo crees?** (Votación rápida.)

> **🇪🇸 Traducción al español**
>
> ## Paso 1 · La afirmación
> > *"Nuestra red neuronal (DeepOnco-MLP) detecta tumores malignos con **100 % de exactitud**. Equilibramos las clases copiando los casos minoritarios (sobremuestreo), escalamos los datos, conservamos las 10 mejores variables y evaluamos con el 20 % de los datos (semilla 12)."*
>
> Suena cuidadoso. **¿Lo crees?** (Votación rápida.)

In [ ]:
# Reproduce el método del "artículo" de principio a fin con una semilla dada y devuelve la exactitud
def paper_pipeline(seed):
    # Generador aleatorio reproducible a partir de la semilla
    rng = np.random.default_rng(seed)
    # 1) Copy minority patients to balance classes ... BEFORE splitting  (problem!)
    #    Copiar pacientes minoritarios ... ANTES de dividir               (¡problema!)
    # Índices de los pacientes malignos (clase minoritaria)
    minority = np.where(y == 0)[0]
    # Sortea con reemplazo tantos malignos como falten para igualar a los benignos
    extra = rng.choice(minority, (y == 1).sum() - len(minority), replace=True)
    # Añade las copias a los datos: Xo (variables) y yo (etiquetas) ya sobremuestreados
    Xo, yo = np.vstack([X, X[extra]]), np.concatenate([y, y[extra]])
    # 2) Scale and select features using ALL rows ... BEFORE splitting  (problem!)
    #    Escalar y seleccionar con TODAS las filas ... ANTES de dividir (¡problema!)
    # Estandariza con todas las filas (fuga de datos)
    Xo = StandardScaler().fit_transform(Xo)
    # Elige las 10 mejores variables con todas las filas (otra fuga de datos)
    Xo = SelectKBest(f_classif, k=10).fit_transform(Xo, yo)
    # 3) One 80/20 split with one seed / Una sola división 80/20 con una semilla
    # Divide 80 % entrenamiento / 20 % prueba (las copias pueden quedar en ambos lados)
    Xtr, Xte, ytr, yte = train_test_split(Xo, yo, test_size=0.2, random_state=seed)
    # Entrena la red neuronal (capas de 64 y 32 neuronas) con la misma semilla
    m = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=2000, random_state=seed).fit(Xtr, ytr)
    # Devuelve la exactitud en el conjunto de prueba
    return accuracy_score(yte, m.predict(Xte))

# Re-ejecuta el método con la semilla 12 del artículo e imprime la exactitud
print(f"Step 2 · Re-run with seed 12 / Re-ejecución con semilla 12: {paper_pipeline(12):.1%}")

## Step 2 · It reproduces! · ¡Se reproduce!
**EN:** Same code, same data, same seed, same number. The result is **reproducible**. But is it **true**? Reproducible only means "I got the same number again". It does not mean "the number is correct".  
**ES:** Mismo código, datos y semilla: mismo número. El resultado es **reproducible**. ¿Pero es **verdadero**? Reproducible solo significa "obtuve el mismo número otra vez". No significa "el número es correcto".

## Step 3 · Check for problems · Buscar problemas
**EN:** A good reviewer asks three simple questions:  
**ES:** Un buen revisor hace tres preguntas simples:

- **(a)** Was seed 12 a lucky choice? · ¿La semilla 12 fue una elección con suerte?
- **(b)** Can the same patient be in both training and test? · ¿Puede el mismo paciente estar en entrenamiento y en prueba?
- **(c)** Did any step "see" the test data? · ¿Algún paso "vio" los datos de prueba?

> **🇪🇸 Traducción al español**
>
> ## Paso 2 · ¡Se reproduce!
> Mismo código, mismos datos, misma semilla, mismo número. El resultado es **reproducible**. ¿Pero es **verdadero**? Reproducible solo significa "obtuve el mismo número otra vez". No significa "el número es correcto".
>
> ## Paso 3 · Buscar problemas
> Un buen revisor hace tres preguntas simples:
>
> - **(a)** ¿La semilla 12 fue una elección con suerte?
> - **(b)** ¿Puede el mismo paciente estar en entrenamiento y en prueba?
> - **(c)** ¿Algún paso "vio" los datos de prueba?

In [ ]:
# (a) Try the paper's exact method with 20 seeds / Probar el método exacto con 20 semillas
# Ejecuta el método del artículo con las semillas 0 a 19 y guarda las exactitudes en una Serie (índice = semilla)
res = pd.Series({s: paper_pipeline(s) for s in range(20)})
# Imprime la exactitud de cada semilla redondeada a 3 decimales
print(res.round(3).to_string())
# Indica si la semilla 12 es justo la de mayor exactitud, y muestra el promedio y el peor resultado
print(f"\nIs seed 12 the best? / ¿La semilla 12 es la mejor?: {res.idxmax() == 12}")
# Imprime el promedio de las 20 semillas y el peor resultado
print(f"Average / Promedio: {res.mean():.1%}   Worst / Peor: {res.min():.1%}")

In [ ]:
# (b) Same patient in train AND test? / ¿Mismo paciente en entrenamiento Y prueba?
# Repite el mismo sorteo de copias del sobremuestreo con la semilla 12
rng = np.random.default_rng(12)
# Índices de los pacientes malignos
minority = np.where(y == 0)[0]
# Pacientes malignos copiados con reemplazo
extra = rng.choice(minority, (y == 1).sum() - len(minority), replace=True)
# Identificador del paciente real detrás de cada fila (originales + copias)
patient_id = np.concatenate([np.arange(len(y)), extra])     # real patient behind each row
# Divide los índices de filas 80/20 igual que el artículo
tr_idx, te_idx = train_test_split(np.arange(len(patient_id)), test_size=0.2, random_state=12)
# Pacientes que aparecen tanto en entrenamiento como en prueba
shared = set(patient_id[tr_idx]) & set(patient_id[te_idx])
# Marca las filas de prueba cuyo paciente también está en entrenamiento
dup = np.array([p in shared for p in patient_id[te_idx]])
# Imprime el total de filas de prueba y cuántas están duplicadas en entrenamiento
print(f"Test rows / Filas de prueba: {len(te_idx)}")
# Imprime cuántas filas de prueba tienen a su paciente también en entrenamiento y su porcentaje
print(f"Test rows whose patient is ALSO in training / Filas cuyo paciente TAMBIÉN está en entrenamiento: {dup.sum()} ({dup.mean():.0%})")

**(c) EN:** Scaling and feature selection used all rows, including the test rows. This is **leakage**: like seeing the exam before the exam.  
**ES:** El escalado y la selección de variables usaron todas las filas, incluidas las de prueba. Esto es **fuga**: como ver el examen antes del examen.

**Verdict · Veredicto:** three problems · tres problemas
1. Lucky seed · Semilla con suerte
2. Same patients in train and test · Mismos pacientes en entrenamiento y prueba
3. Leakage · Fuga de datos

## Step 4 · Fix and re-run · Corregir y volver a ejecutar
**EN:** Three simple rules:  
**ES:** Tres reglas simples:
- Put every step that learns from data inside a `Pipeline`. · Todo paso que aprende de los datos va dentro de un `Pipeline`.
- Test many times (5-fold cross-validation, repeated 10 times = 50 tests). · Evaluar muchas veces (validación cruzada 5 particiones × 10 = 50 pruebas).
- Always compare with a **simple model** and a **"dumb" model**. · Comparar siempre con un **modelo simple** y un **modelo "tonto"**.

> **🇪🇸 Traducción al español**
>
> **(c)** El escalado y la selección de variables usaron todas las filas, incluidas las de prueba. Esto es **fuga de datos**: como ver el examen antes del examen.
>
> **Veredicto:** tres problemas
> 1. Semilla con suerte
> 2. Mismos pacientes en entrenamiento y prueba
> 3. Fuga de datos
>
> ## Paso 4 · Corregir y volver a ejecutar
> Tres reglas simples:
> - Todo paso que aprende de los datos va dentro de un `Pipeline`.
> - Evaluar muchas veces (validación cruzada de 5 particiones, repetida 10 veces = 50 pruebas).
> - Comparar siempre con un **modelo simple** y un **modelo "tonto"**.

In [ ]:
# Validación cruzada de 5 particiones repetida 10 veces (50 pruebas) con semilla fija
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=2026)
# Diccionario de modelos a comparar; todo paso que aprende va dentro de un Pipeline
models = {
    # Modelo tonto: siempre predice la clase mayoritaria ("benigno")
    "Dumb model: always guess 'benign' / Modelo tonto": DummyClassifier(strategy="most_frequent"),
    # Modelo simple: regresión logística con escalado dentro del Pipeline
    "Simple model: logistic regression / Modelo simple": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    # Red neuronal bien hecha: escalado + selección de 10 variables + MLP, todo dentro del Pipeline
    "DeepOnco-MLP, done right / bien hecho": make_pipeline(StandardScaler(), SelectKBest(f_classif, k=10),
                                       # Último paso del Pipeline: red neuronal (64 y 32 neuronas) con semilla fija; cierra make_pipeline y la entrada del diccionario
                                       MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=2000, random_state=0)),
# Cierra el diccionario de modelos
}
# Lista para acumular una fila de resultados por modelo
rows = []
# Evalúa cada modelo con las mismas particiones
for name, m in models.items():
    # Obtiene las 50 exactitudes
    s = cross_val_score(m, X, y, cv=cv)
    # Guarda media, desviación estándar muestral y número de pruebas
    rows.append({"Model / Modelo": name, "Mean / Media": s.mean(), "SD / DE": s.std(ddof=1), "Tests": len(s)})
# Construye la tabla de resultados indexada por modelo
report = pd.DataFrame(rows).set_index("Model / Modelo")
# Imprime la tabla redondeada a 3 decimales
print(report.round(3).to_string())

In [ ]:
# Importa matplotlib para graficar
import matplotlib.pyplot as plt
# Figura con un solo eje de 8x3.2 pulgadas
fig, ax = plt.subplots(figsize=(8, 3.2))
# Etiquetas de las cinco barras
labels = ["Paper claim\n(seed 12)", "Paper method\n(20 seeds)", "MLP\ndone right", "Logistic\nregression", "Dumb\nmodel"]
# Valores: afirmación (semilla 12), media del método del artículo, MLP honesto, regresión logística y modelo tonto
vals = [res[12], res.mean(), report.iloc[2, 0], report.iloc[1, 0], report.iloc[0, 0]]
# Un color por barra
cols = ["#E07A1F", "#F2B179", "#0E9F9A", "#0B1F3A", "#8A9AAD"]
# Dibuja las barras en porcentaje
ax.bar(labels, np.array(vals) * 100, color=cols)
# Escribe el valor sobre cada barra
for i, v in enumerate(vals): ax.text(i, v * 100 + 0.8, f"{v:.1%}", ha="center", fontsize=9)
# Límites y título del eje Y
ax.set_ylim(50, 105); ax.set_ylabel("Accuracy (%)")
# Título del gráfico
ax.set_title("From claim to honest number · De la afirmación al número honesto")
# Ajusta el diseño y muestra la figura
plt.tight_layout(); plt.show()

## Step 5 · Honest report · Informe honesto
**EN:** Two lessons. (1) The MLP is still good, but a **simple logistic regression is just as good or better**. (2) Always report **mean ± SD**, never one lucky number.  
**ES:** Dos lecciones. (1) El MLP sigue siendo bueno, pero una **regresión logística simple es igual o mejor**. (2) Reporta siempre **media ± DE**, nunca un número con suerte.

> **🇪🇸 Traducción al español**
>
> ## Paso 5 · Informe honesto
> Dos lecciones. (1) El MLP sigue siendo bueno, pero una **regresión logística simple es igual de buena o mejor**. (2) Reporta siempre **media ± DE**, nunca un solo número con suerte.

In [ ]:
# Fila de resultados de la regresión logística y del MLP honesto
lr = report.iloc[1]; mlp = report.iloc[2]
# Encabezado del informe
print("REPRODUCTION REPORT · INFORME DE REPRODUCCIÓN")
# Línea separadora
print("-" * 60)
# ¿Se obtuvo el mismo número con la semilla 12?
print(f"Same number again? / ¿Mismo número?    : YES / SÍ, {res[12]:.1%}")
# ¿Es válida la afirmación? No
print(f"Is the claim valid? / ¿Es válida?      : NO")
# Problemas encontrados
print(f"Problems / Problemas                   : lucky seed; duplicate patients; leakage")
# Estimación honesta del MLP: media ± DE
print(f"Honest MLP / MLP honesto               : {mlp.iloc[0]:.1%} ± {mlp.iloc[1]:.1%}")
# Estimación del modelo simple: media ± DE
print(f"Simple model / Modelo simple           : {lr.iloc[0]:.1%} ± {lr.iloc[1]:.1%}")
# Entorno: versiones de Python y scikit-learn
print(f"Environment / Entorno                  : Python {platform.python_version()}, scikit-learn {sklearn.__version__}")
# Línea separadora
print("-" * 60)
# Conclusión: reproducible pero no válido
print("EN: Reproducible, but not valid.   ES: Reproducible, pero no válido.")

## 🎯 Homework · Tarea
1. **EN:** Replace the MLP with your own thesis model. Which simple model should you compare against? · **ES:** Reemplaza el MLP por el modelo de tu tesis. ¿Con qué modelo simple deberías compararlo?
2. **EN:** Pick one published paper in your field and write the 3-line report: *Same number? Valid? Problems?* · **ES:** Elige un artículo de tu área y escribe el informe de 3 líneas: *¿Mismo número? ¿Válido? ¿Problemas?*

**Reading · Lectura:** Kapoor & Narayanan (2023), *Patterns* · [doi.org/10.1016/j.patter.2023.100804](https://doi.org/10.1016/j.patter.2023.100804)

> **🇪🇸 Traducción al español**
>
> ## 🎯 Tarea
> 1. Reemplaza el MLP por el modelo de tu tesis. ¿Con qué modelo simple deberías compararlo?
> 2. Elige un artículo publicado en tu área y escribe el informe de 3 líneas: *¿Mismo número? ¿Válido? ¿Problemas?*
>
> **Lectura:** Kapoor & Narayanan (2023), *Patterns* · [doi.org/10.1016/j.patter.2023.100804](https://doi.org/10.1016/j.patter.2023.100804)